# TB Control · AI Inference Server (Kaggle)

Запускает vLLM + YOLO + Cloudflare Tunnel на 2× T4.
Цель: публичный HTTPS endpoint для нашего Next.js приложения.

**Перед запуском:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Run All

После загрузки моделей (~5 мин) последняя ячейка выведет публичный URL.
Скопируй его в `.env.local` нашего app:
```
NEXT_PUBLIC_VAST_INFERENCE_URL=https://xxxxx.trycloudflare.com
NEXT_PUBLIC_USE_MOCK_INFERENCE=false
```

## 1. Setup — install dependencies

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!pip install -q vllm==0.6.4 ultralytics fastapi uvicorn python-multipart 2>&1 | tail -5

In [ ]:
# Cloudflared binary — gives us free *.trycloudflare.com tunnel
import os, subprocess
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 2. Download models (Qwen 2.5-7B + Qwen-VL 7B AWQ)

Use 7B for both — fits comfortably on a single T4, leaves the other free for Vision.
Total ~10 GB downloaded in 1-2 minutes from HuggingFace.

In [ ]:
import os
os.makedirs('/kaggle/working/models', exist_ok=True)

from huggingface_hub import snapshot_download

# LLM for AI assistant chat — 7B AWQ fits in ~4 GB VRAM
llm_path = snapshot_download(
    repo_id='Qwen/Qwen2.5-7B-Instruct-AWQ',
    local_dir='/kaggle/working/models/llm',
    local_dir_use_symlinks=False,
)
print(f'LLM ready: {llm_path}')

# Vision LLM for pill closeup verification — 7B AWQ ~5 GB VRAM
vision_path = snapshot_download(
    repo_id='Qwen/Qwen2-VL-7B-Instruct-AWQ',
    local_dir='/kaggle/working/models/vision',
    local_dir_use_symlinks=False,
)
print(f'Vision ready: {vision_path}')

!du -sh /kaggle/working/models/*

## 3. Start vLLM servers in background

- LLM (port 8001) on GPU 0
- Vision (port 8002) on GPU 1
- Both behind a single FastAPI router on port 7860 → cloudflared tunnel

In [ ]:
import subprocess, time, os

log_dir = '/kaggle/working/logs'
os.makedirs(log_dir, exist_ok=True)

# LLM on GPU 0
llm_cmd = (
    'CUDA_VISIBLE_DEVICES=0 vllm serve /kaggle/working/models/llm '
    '--served-model-name davoai-llm '
    '--host 0.0.0.0 --port 8001 '
    '--max-model-len 4096 '
    '--gpu-memory-utilization 0.85 '
    '--quantization awq_marlin '
    '--enforce-eager'
)
subprocess.Popen(['bash', '-c', f'{llm_cmd} > {log_dir}/llm.log 2>&1'])
print('LLM starting on port 8001 (GPU 0)…')

# Vision on GPU 1
vision_cmd = (
    'CUDA_VISIBLE_DEVICES=1 vllm serve /kaggle/working/models/vision '
    '--served-model-name davoai-vision '
    '--host 0.0.0.0 --port 8002 '
    '--max-model-len 4096 '
    '--gpu-memory-utilization 0.85 '
    '--quantization awq_marlin '
    '--limit-mm-per-prompt \'{"image": 2}\' '
    '--enforce-eager'
)
subprocess.Popen(['bash', '-c', f'{vision_cmd} > {log_dir}/vision.log 2>&1'])
print('Vision starting on port 8002 (GPU 1)…')

In [ ]:
# Wait for both to be ready (typically 60-90 sec for AWQ load)
import time, requests

def wait_ready(port, name, timeout=240):
    start = time.time()
    while time.time() - start < timeout:
        try:
            r = requests.get(f'http://localhost:{port}/v1/models', timeout=2)
            if r.status_code == 200:
                print(f'✓ {name} ready ({int(time.time() - start)}s)')
                return True
        except Exception:
            pass
        time.sleep(3)
    print(f'✗ {name} not ready after {timeout}s — check logs!')
    return False

wait_ready(8001, 'LLM')
wait_ready(8002, 'Vision')

## 4. YOLO pill detector (port 8004)

Default Ultralytics YOLOv8m on COCO classes — replace with custom-trained
Ascorutin model when ready. For demo this still triggers bbox detection on
objects, and the dose-flow falls back to mock visualization if class doesn't match.

In [ ]:
yolo_server_code = '''
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from ultralytics import YOLO
from PIL import Image
import io, time

app = FastAPI()
model = YOLO("yolov8m.pt")  # COCO pretrained — swap with custom Ascorutin weights

@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": True, "using_fallback_weights": True}

@app.post("/detect")
async def detect(image: UploadFile = File(...)):
    t0 = time.time()
    img = Image.open(io.BytesIO(await image.read()))
    results = model(img, verbose=False)
    detections = []
    for r in results:
        if r.boxes is None: continue
        for b in r.boxes:
            x1, y1, x2, y2 = b.xyxyn[0].tolist()
            detections.append({
                "label": model.names[int(b.cls[0])],
                "confidence": float(b.conf[0]),
                "bbox": [x1, y1, x2, y2],
            })
    return {"detections": detections, "inferenceMs": int((time.time() - t0) * 1000)}
'''
with open('/kaggle/working/yolo_server.py', 'w') as f:
    f.write(yolo_server_code)

subprocess.Popen([
    'bash', '-c',
    'cd /kaggle/working && uvicorn yolo_server:app --host 0.0.0.0 --port 8004 '
    f'> {log_dir}/yolo.log 2>&1',
])
print('YOLO starting on port 8004…')
wait_ready(8004, 'YOLO', timeout=60) or print('Check log:', log_dir + '/yolo.log')

## 5. Reverse proxy: route 8001/8002/8004 through one port (7860)

Cloudflare's free tunnel gives ONE URL per tunnel. We multiplex all three
services behind a tiny FastAPI proxy.

In [ ]:
proxy_code = '''
from fastapi import FastAPI, Request, Response
from fastapi.middleware.cors import CORSMiddleware
import httpx

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
)

client = httpx.AsyncClient(timeout=120.0)

ROUTES = {
    "/llm": "http://localhost:8001",     # → LLM chat (Qwen 7B)
    "/vision": "http://localhost:8002",  # → Vision (Qwen-VL 7B)
    "/yolo": "http://localhost:8004",    # → YOLO
}

@app.get("/")
async def root():
    return {"davoai": "ok", "routes": list(ROUTES.keys())}

@app.api_route("/{prefix}/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS"])
async def proxy(prefix: str, path: str, request: Request):
    target = ROUTES.get(f"/{prefix}")
    if not target:
        return Response(content=f"unknown prefix: /{prefix}", status_code=404)
    url = f"{target}/{path}"
    body = await request.body()
    headers = {k: v for k, v in request.headers.items() if k.lower() not in ("host", "content-length")}
    r = await client.request(request.method, url, content=body, headers=headers, params=request.query_params)
    return Response(content=r.content, status_code=r.status_code, headers=dict(r.headers))
'''
with open('/kaggle/working/proxy.py', 'w') as f:
    f.write(proxy_code)

subprocess.Popen([
    'bash', '-c',
    'cd /kaggle/working && uvicorn proxy:app --host 0.0.0.0 --port 7860 '
    f'> {log_dir}/proxy.log 2>&1',
])
time.sleep(3)
import requests
print(requests.get('http://localhost:7860/').json())

## 6. Open Cloudflare Tunnel — get public URL

This prints a **public HTTPS URL** like `https://abc-def-ghi.trycloudflare.com`.
Copy it into your app's `.env.local`.

In [ ]:
import subprocess, re, time

# Spawn tunnel
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, universal_newlines=True,
)

tunnel_url = None
for _ in range(60):
    line = proc.stdout.readline()
    if not line: time.sleep(0.5); continue
    print(line.rstrip())
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break

print('\n' + '=' * 70)
if tunnel_url:
    print(f'✅ PUBLIC URL: {tunnel_url}')
    print('\nAdd to your local .env.local:')
    print(f'NEXT_PUBLIC_VAST_INFERENCE_URL={tunnel_url}')
    print('NEXT_PUBLIC_USE_MOCK_INFERENCE=false')
    print('\nEndpoints:')
    print(f'  {tunnel_url}/llm/v1/chat/completions  → Qwen 7B chat')
    print(f'  {tunnel_url}/vision/v1/chat/completions → Qwen-VL 7B vision')
    print(f'  {tunnel_url}/yolo/detect → YOLO pill detection')
else:
    print('✗ Tunnel did not come up — check log above')
print('=' * 70)

## 7. Keep-alive (prevent 20-min idle kill)

Pings own LLM endpoint every 12 minutes. Run this and **leave the cell running**
for the entire demo session (it never finishes by itself).

In [ ]:
import time, requests
from datetime import datetime

while True:
    try:
        r = requests.get('http://localhost:7860/', timeout=5)
        print(f'[{datetime.now().strftime("%H:%M:%S")}] alive · {r.status_code}')
    except Exception as e:
        print(f'[{datetime.now().strftime("%H:%M:%S")}] err: {e}')
    time.sleep(60 * 12)  # ping every 12 min (kaggle kills at 20)